# R=16-Budget Mild Late-Ramp LoRA Experiment

This notebook runs the `r=16`-budget version of the best `r=4` depth-allocation pattern:

1. `late_ramp_balanced_r16_budget`: scaled version of the original mild late ramp.

The scalar config uses `lora.rank=16` and `lora.alpha=32` to match the existing uniform rank sweep convention. This run has the same trainable-parameter budget as uniform `r=16`: `1,572,864` trainable LoRA parameters.

The goal is to test whether a same-budget late-layer allocation can beat uniform `r=16` on test BLEU/ROUGE. The run is backed up to Google Drive immediately after training, after generation/evaluation, and after the official E2E scorer.

## 1. Check GPU

Run this first to confirm Colab assigned a CUDA GPU. Prefer H100 or A100 for this run if available.

In [ ]:
!nvidia-smi

import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 2. Clone, Install, Drive, And Setup

Run these cells from a clean Colab runtime. They clone or update the project under `/content/CS4782-final-project`, switch into `code`, install dependencies, mount Google Drive, and load the libraries used by the `r=16` late-ramp experiment.

In [ ]:
from getpass import getpass
from pathlib import Path
import os
import subprocess

REPO_OWNER = 'justinlxiang'
REPO_NAME = 'CS4782-final-project'
BRANCH = 'main'
PROJECT_DIR = Path('/content') / REPO_NAME
WORK_DIR = PROJECT_DIR / 'code'

token = getpass('GitHub token, or press Enter for public clone: ')
repo_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
if token:
    repo_url = f'https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git'

if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, repo_url, str(PROJECT_DIR)], check=True)

os.chdir(WORK_DIR)
print('working directory:', Path.cwd())
!git log --oneline -3

In [ ]:
!pip install -q -r requirements.txt

import copy
import json
import os
import shlex
import shutil
import subprocess
from collections import deque
from pathlib import Path

from google.colab import drive
import matplotlib.pyplot as plt
import pandas as pd
import yaml

# Keep full run artifacts backed up in Drive, separate from the r=4 and r=8 pattern notebooks.
drive.mount('/content/drive')
DRIVE_PATTERN_DIR = Path('/content/drive/MyDrive/lora_late_ramp_r16_budget')
DRIVE_PATTERN_DIR.mkdir(parents=True, exist_ok=True)
print('Drive pattern dir:', DRIVE_PATTERN_DIR)

!python -m pytest tests/test_lora_layers.py tests/test_inject.py tests/test_checkpointing.py

## 3. Data Prep

This matches the existing E2E preprocessing path used by the uniform rank sweep and rank-pattern notebooks.

In [ ]:
!mkdir -p ../data/raw/e2e
!curl -L -o ../data/raw/e2e/train.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/train.txt
!curl -L -o ../data/raw/e2e/valid.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/valid.txt
!curl -L -o ../data/raw/e2e/test.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/test.txt
!python scripts/prepare_e2e.py --config configs/e2e_gpt2_medium_lora.yaml
!wc -l ../data/raw/e2e/*.txt ../data/processed/e2e_gpt2/*.jsonl

## 4. Pattern Matrix

This is the `r=16`-budget version of the original mild late-ramp allocation. It scales each rank from the `r=4`-budget counterpart by 4.

For GPT-2 Medium, one Q or V rank unit costs `1024 + 1024 = 2048` trainable parameters. The total `r=16` budget is `768` rank units, so the run has `768 * 2048 = 1,572,864` trainable parameters.

| Run | Layers | `r_q` | `r_v` | Total rank units |
| --- | --- | ---: | ---: | ---: |
| `late_ramp_balanced_r16_budget` | 0-7 | 8 | 8 | |
| `late_ramp_balanced_r16_budget` | 8-15 | 16 | 16 | |
| `late_ramp_balanced_r16_budget` | 16-23 | 24 | 24 | 768 |

In [ ]:
BASELINE_TRAINABLE_PARAMS = 1_572_864
BASELINE_RANK = 16
BASELINE_ALPHA = 32
TOTAL_RANK_UNITS = 768
GENERATION_BATCH_SIZE = 16
MAX_TRAIN_STEPS = None  # Set to 2000 for a pilot; None means full 5 epochs.
RUN_TRAINING = True
RUN_GENERATION_AND_EVAL = True
RUN_OFFICIAL_E2E = True
SUMMARY_STEM = 'late_ramp_r16_budget_summary'
NOTEBOOK_NAME = 'colab_late_ramp_r16_budget_lora.ipynb'


def repeated_pattern(blocks):
    pattern = []
    for count, q_rank, v_rank in blocks:
        pattern.extend({'query': q_rank, 'value': v_rank} for _ in range(count))
    assert len(pattern) == 24
    return pattern


PATTERNS = [
    {
        'comparison': 'late_ramp_r16_budget',
        'pattern_name': 'late_ramp_balanced_r16_budget',
        'description': 'Balanced Q/V; r=16-budget version of the original mild late ramp.',
        'rank_pattern': repeated_pattern([(8, 8, 8), (8, 16, 16), (8, 24, 24)]),
    },
]

for spec in PATTERNS:
    rank_units = sum(layer['query'] + layer['value'] for layer in spec['rank_pattern'])
    params = rank_units * (1024 + 1024)
    print(spec['comparison'], spec['pattern_name'], 'rank_units=', rank_units, 'params=', params)
    assert rank_units == TOTAL_RANK_UNITS
    assert params == BASELINE_TRAINABLE_PARAMS

## 5. Create Pattern Config

The generated config keeps scalar `lora.rank=16` and `lora.alpha=32` to match the uniform `r=16` rank-sweep setting. The `lora.rank_pattern` field overrides Q/V ranks per layer.

In [ ]:
base_config_path = Path('configs/e2e_gpt2_medium_lora.yaml')
base_config = yaml.safe_load(base_config_path.read_text())
pattern_config_paths = []

for spec in PATTERNS:
    cfg = copy.deepcopy(base_config)
    run_name = spec['pattern_name']
    run_dir = Path('outputs/runs/rank_patterns') / run_name

    cfg['project']['output_dir'] = str(run_dir)
    cfg['lora']['rank'] = BASELINE_RANK
    cfg['lora']['alpha'] = BASELINE_ALPHA
    cfg['lora']['rank_pattern'] = spec['rank_pattern']
    cfg['generation']['decoder'] = 'official_beam'
    cfg['generation']['length_penalty'] = 0.9
    cfg['generation']['batch_size'] = GENERATION_BATCH_SIZE
    cfg['evaluation']['predictions_file'] = str(run_dir / 'generations_test.txt')
    cfg['evaluation']['references_file'] = '../data/processed/e2e_gpt2/references_test.txt'

    out_path = Path('configs/rank_patterns') / f"e2e_gpt2_medium_lora_{run_name}.yaml"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    pattern_config_paths.append((spec, out_path, run_dir))

for spec, config_path, run_dir in pattern_config_paths:
    print(spec['comparison'], spec['pattern_name'], config_path, '->', run_dir)

for spec, config_path, run_dir in pattern_config_paths:
    print('\n===', spec['pattern_name'], 'parameter count ===')
    result = subprocess.run(
        ['python', 'scripts/count_params.py', '--config', str(config_path)],
        text=True,
        capture_output=True,
        check=True,
    )
    print(result.stdout)
    assert f'expected_gpt2_lora_parameters={BASELINE_TRAINABLE_PARAMS}' in result.stdout

## 6. Train, Generate, And Evaluate

The run is copied to Google Drive immediately after training, after generation/evaluation, and after the official E2E scorer so completed work survives Colab runtime resets.

In [ ]:
def persist_pattern_run_to_drive(spec, config_path, run_dir, phase):
    DRIVE_PATTERN_DIR.mkdir(parents=True, exist_ok=True)
    pattern_name = spec['pattern_name']

    if config_path.exists():
        config_dest = DRIVE_PATTERN_DIR / 'configs' / config_path.name
        config_dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(config_path, config_dest)
        print(f"[{phase}] Backed up config for {pattern_name}: {config_dest}")

    if run_dir.exists():
        run_dest = DRIVE_PATTERN_DIR / run_dir.name
        shutil.copytree(run_dir, run_dest, dirs_exist_ok=True)
        print(f"[{phase}] Backed up run for {pattern_name}: {run_dest}")
    else:
        print(f"[{phase}] No run directory yet for {pattern_name}: {run_dir}")


def run_checked(cmd, env=None):
    print('+', shlex.join(cmd), flush=True)
    tail = deque(maxlen=80)
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        tail.append(line)
    returncode = process.wait()
    if returncode != 0:
        print('\nCommand failed:', shlex.join(cmd))
        print('Last output lines:')
        print(''.join(tail))
        raise subprocess.CalledProcessError(returncode, cmd)


if RUN_TRAINING:
    for spec, config_path, run_dir in pattern_config_paths:
        print(f"=== Training {spec['pattern_name']} ===")
        cmd = ['python', 'scripts/train.py', '--config', str(config_path), '--train', '--device', 'cuda']
        if MAX_TRAIN_STEPS is not None:
            cmd += ['--max-train-steps', str(MAX_TRAIN_STEPS)]
        run_checked(cmd)
        persist_pattern_run_to_drive(spec, config_path, run_dir, phase='training')
else:
    print('RUN_TRAINING=False; skipping training.')

In [ ]:
if RUN_GENERATION_AND_EVAL:
    env = {**os.environ, 'TOKENIZERS_PARALLELISM': 'false', 'TRANSFORMERS_VERBOSITY': 'error'}
    for spec, config_path, run_dir in pattern_config_paths:
        print(f"\n=== Generating/evaluating {spec['pattern_name']} ===")
        adapter = run_dir / 'checkpoints' / 'adapter_final.pt'
        if not adapter.exists():
            raise FileNotFoundError(f'Missing adapter: {adapter}')

        run_checked([
            'python', 'scripts/generate.py',
            '--config', str(config_path),
            '--split', 'test',
            '--adapter', str(adapter),
            '--batch-size', str(GENERATION_BATCH_SIZE),
        ], env=env)
        run_checked(['python', 'scripts/evaluate.py', '--config', str(config_path)], env=env)
        run_checked([
            'python', 'scripts/make_figures.py',
            '--config', str(config_path),
            '--run-dir', str(run_dir),
            '--figures-dir', str(run_dir / 'figures'),
        ], env=env)
        persist_pattern_run_to_drive(spec, config_path, run_dir, phase='generation/eval')
else:
    print('RUN_GENERATION_AND_EVAL=False; skipping generation/evaluation.')

In [ ]:
if RUN_OFFICIAL_E2E:
    Path('external').mkdir(exist_ok=True)
    if not Path('external/e2e-metrics/.git').exists():
        run_checked(['git', 'clone', 'https://github.com/tuetschek/e2e-metrics.git', 'external/e2e-metrics'])

    for spec, config_path, run_dir in pattern_config_paths:
        ref_file = run_dir / 'generations_test.e2e_refs.txt'
        pred_file = run_dir / 'generations_test.e2e_preds.txt'
        out_file = run_dir / 'generations_test.official_e2e_metrics.txt'
        print(f"\n=== Official E2E metrics {spec['pattern_name']} ===")
        result = subprocess.run(
            ['python', 'external/e2e-metrics/measure_scores.py', str(ref_file), str(pred_file), '-p'],
            text=True,
            capture_output=True,
        )
        out_file.write_text('STDOUT:\n' + result.stdout + '\n\nSTDERR:\n' + result.stderr, encoding='utf-8')
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
        persist_pattern_run_to_drive(spec, config_path, run_dir, phase='official-e2e')
else:
    print('RUN_OFFICIAL_E2E=False; skipping official scorer.')

## 7. Summary

The summary includes uniform `r=16` rank-sweep anchor plus the new `r=16`-budget mild late-ramp run.

In [ ]:
def read_validation_summary(run_dir):
    metrics_path = run_dir / 'metrics.jsonl'
    validation = []
    if metrics_path.exists():
        for line in metrics_path.read_text().splitlines():
            if line.strip():
                record = json.loads(line)
                if 'valid_loss' in record:
                    validation.append(record)
    best = min(validation, key=lambda row: row['valid_loss']) if validation else {}
    final = validation[-1] if validation else {}
    return best, final


def add_rank_sweep_rows(rows):
    for summary_path in [Path('outputs/rank_sweep_summary.csv'), Path('outputs/runs/rank_sweep_summary.csv')]:
        if summary_path.exists():
            sweep_df = pd.read_csv(summary_path)
            for rank in [16]:
                matches = sweep_df[sweep_df['rank'] == rank]
                if matches.empty:
                    continue
                row = matches.iloc[0].to_dict()
                rows.append({
                    'comparison': 'uniform_rank_sweep',
                    'pattern_name': f'uniform_r{rank}',
                    'total_rank_units': 24 * 2 * rank,
                    'trainable_params': row.get('trainable_params'),
                    'best_valid_loss': row.get('best_valid_loss'),
                    'best_valid_ppl': row.get('best_valid_ppl'),
                    'best_valid_nll_loss': row.get('best_valid_nll_loss'),
                    'best_valid_nll_ppl': row.get('best_valid_nll_ppl'),
                    'best_valid_epoch': row.get('best_valid_epoch'),
                    'final_valid_loss': row.get('final_valid_loss'),
                    'final_valid_nll_loss': row.get('final_valid_nll_loss'),
                    'bleu': row.get('bleu'),
                    'rouge_l': row.get('rouge_l'),
                    'line_bleu': row.get('line_bleu'),
                    'line_rouge_l': row.get('line_rouge_l'),
                    'run_dir': row.get('run_dir'),
                })
            return


rows = []
add_rank_sweep_rows(rows)

for spec, config_path, run_dir in pattern_config_paths:
    metrics_path = run_dir / 'generations_test.metrics.json'
    params_path = run_dir / 'parameter_report.json'
    metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else {}
    params = json.loads(params_path.read_text()) if params_path.exists() else {}
    best, final = read_validation_summary(run_dir)
    rank_units = sum(layer['query'] + layer['value'] for layer in spec['rank_pattern'])
    rows.append({
        'comparison': spec['comparison'],
        'pattern_name': spec['pattern_name'],
        'total_rank_units': rank_units,
        'trainable_params': params.get('trainable'),
        'best_valid_loss': best.get('valid_loss'),
        'best_valid_ppl': best.get('valid_ppl'),
        'best_valid_nll_loss': best.get('valid_nll_loss'),
        'best_valid_nll_ppl': best.get('valid_nll_ppl'),
        'best_valid_epoch': best.get('epoch'),
        'final_valid_loss': final.get('valid_loss'),
        'final_valid_nll_loss': final.get('valid_nll_loss'),
        'bleu': metrics.get('bleu'),
        'rouge_l': metrics.get('rouge_l'),
        'line_bleu': metrics.get('line_bleu'),
        'line_rouge_l': metrics.get('line_rouge_l'),
        'run_dir': str(run_dir),
    })

df = pd.DataFrame(rows)
summary_path = Path('outputs/runs') / f'{SUMMARY_STEM}.csv'
summary_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(summary_path, index=False)
display(df)
print('Saved:', summary_path)

plot_df = df[df['bleu'].notna()].copy()
if not plot_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(plot_df['pattern_name'], plot_df['bleu'])
    ax.set_title('R=16-Budget Late-Ramp BLEU')
    ax.set_ylabel('BLEU')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    plot_path = Path('outputs/runs') / f'{SUMMARY_STEM}.png'
    fig.savefig(plot_path, dpi=180)
    print('Saved:', plot_path)

## 8. Back Up R=16 Run To Drive

Copy the completed pattern run, generated config, summary artifact, and this notebook into Google Drive.

In [ ]:
DRIVE_PATTERN_DIR.mkdir(parents=True, exist_ok=True)

for spec, config_path, run_dir in pattern_config_paths:
    if run_dir.exists():
        destination = DRIVE_PATTERN_DIR / run_dir.name
        shutil.copytree(run_dir, destination, dirs_exist_ok=True)
        print('Backed up:', run_dir, '->', destination)
    if config_path.exists():
        config_dest = DRIVE_PATTERN_DIR / 'configs' / config_path.name
        config_dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(config_path, config_dest)
        print('Backed up:', config_path, '->', config_dest)

for artifact_path in [
    Path('outputs/runs') / f'{SUMMARY_STEM}.csv',
    Path('outputs/runs') / f'{SUMMARY_STEM}.png',
    Path(NOTEBOOK_NAME),
]:
    if artifact_path.exists():
        shutil.copy2(artifact_path, DRIVE_PATTERN_DIR / artifact_path.name)
        print('Backed up:', artifact_path)

!find /content/drive/MyDrive/lora_late_ramp_r16_budget -maxdepth 3 -type f | sort | tail -80